In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F

df_level_events_raw = spark.table(
    "lh_bronze_game.level_events_raw"
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 3, Finished, Available, Finished, False)

In [2]:
print("Row count:", df_level_events_raw.count())
print("Column count:", len(df_level_events_raw.columns))

df_level_events_raw.printSchema()

display(df_level_events_raw.limit(5))

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 4, Finished, Available, Finished, False)

Row count: 3741631
Column count: 6
root
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- properties: struct (nullable = true)
 |    |-- attempt_number: long (nullable = true)
 |    |-- booster_used: boolean (nullable = true)
 |    |-- client_build: string (nullable = true)
 |    |-- level_number: long (nullable = true)
 |    |-- moves_remaining: long (nullable = true)
 |    |-- unexpected_debug_flag: boolean (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 7c86dd6b-776d-49da-af36-08aafcd19d5a)

In [3]:
df_level_events_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_level_events_raw.columns
]).show(truncate=False)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 5, Finished, Available, Finished, False)

+--------+----------+---------+----------+----------+---------+
|event_id|event_name|player_id|properties|session_id|timestamp|
+--------+----------+---------+----------+----------+---------+
|0       |0         |0        |0         |3793      |0        |
+--------+----------+---------+----------+----------+---------+



In [4]:
total_events = df_level_events_raw.count()

unique_event_ids = (
    df_level_events_raw
    .select("event_id")
    .distinct()
    .count()
)

print("Total events:", total_events)
print("Unique event_id:", unique_event_ids)
print("Duplicate:", total_events - unique_event_ids)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 6, Finished, Available, Finished, False)

Total events: 3741631
Unique event_id: 3737902
Duplicate: 3729


In [5]:
df_level_events_clean = df_level_events_raw.dropDuplicates()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 7, Finished, Available, Finished, False)

In [6]:
print("Raw:", df_level_events_raw.count())
print("Clean:", df_level_events_clean.count())

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 8, Finished, Available, Finished, False)

Raw: 3741631
Clean: 3737902


In [7]:
df_level_events_clean = (
    df_level_events_clean
    .withColumn(
        "session_id",
        F.when(
            F.col("session_id").isNull(),
            F.lit("Unknown")
        ).otherwise(F.col("session_id"))
    )
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 9, Finished, Available, Finished, False)

In [8]:
df_level_events_clean.filter(
    F.col("session_id").isNull()
).count()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 10, Finished, Available, Finished, False)

0

In [9]:
df_level_events_clean = (
    df_level_events_clean
    .withColumn("attempt_number", F.col("properties.attempt_number"))
    .withColumn("booster_used", F.col("properties.booster_used"))
    .withColumn("client_build", F.col("properties.client_build"))
    .withColumn("level_number", F.col("properties.level_number"))
    .withColumn("moves_remaining", F.col("properties.moves_remaining"))
    .withColumn("unexpected_debug_flag", F.col("properties.unexpected_debug_flag"))
    .drop("properties")
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 11, Finished, Available, Finished, False)

In [10]:
df_time_check = (
    df_level_events_clean
    .withColumn(
        "timestamp_parsed",
        F.to_timestamp("timestamp")
    )
)

df_time_check.select(
    F.count(
        F.when(F.col("timestamp_parsed").isNull(), 1)
    ).alias("invalid_timestamp")
).show()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 12, Finished, Available, Finished, False)

+-----------------+
|invalid_timestamp|
+-----------------+
|                0|
+-----------------+



In [11]:
df_level_events_clean = (
    df_level_events_clean
    .withColumn(
        "timestamp",
        F.to_timestamp("timestamp")
    )
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 13, Finished, Available, Finished, False)

In [12]:
df_level_events_clean.printSchema()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 14, Finished, Available, Finished, False)

root
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- attempt_number: long (nullable = true)
 |-- booster_used: boolean (nullable = true)
 |-- client_build: string (nullable = true)
 |-- level_number: long (nullable = true)
 |-- moves_remaining: long (nullable = true)
 |-- unexpected_debug_flag: boolean (nullable = true)



In [13]:
df_level_events_clean.groupBy("event_name").count().show()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 15, Finished, Available, Finished, False)

+--------------+-------+
|    event_name|  count|
+--------------+-------+
|    level_fail| 462430|
|   level_start|1868951|
|level_complete|1406521|
+--------------+-------+



In [14]:
df_level_events_clean.select(
    F.min("level_number").alias("min_level"),
    F.max("level_number").alias("max_level"),
    F.min("attempt_number").alias("min_attempt"),
    F.max("attempt_number").alias("max_attempt"),
    F.min("moves_remaining").alias("min_moves"),
    F.max("moves_remaining").alias("max_moves")
).show()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 16, Finished, Available, Finished, False)

+---------+---------+-----------+-----------+---------+---------+
|min_level|max_level|min_attempt|max_attempt|min_moves|max_moves|
+---------+---------+-----------+-----------+---------+---------+
|        1|      459|          1|         13|        0|       14|
+---------+---------+-----------+-----------+---------+---------+



In [15]:
df_level_events_clean.groupBy("event_name").count().show()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 17, Finished, Available, Finished, False)

+--------------+-------+
|    event_name|  count|
+--------------+-------+
|    level_fail| 462430|
|   level_start|1868951|
|level_complete|1406521|
+--------------+-------+



In [16]:
orphan_level_players = (
    df_level_events_clean.alias("l")
    .join(
        spark.table("lh_silver_game.players_clean").alias("p"),
        F.col("l.player_id") == F.col("p.player_id"),
        "left_anti"
    )
)

print(
    "Players tablosunda bulunmayan level event:",
    orphan_level_players.count()
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 18, Finished, Available, Finished, False)

Players tablosunda bulunmayan level event: 0


In [17]:
orphan_level_sessions = (
    df_level_events_clean
    .filter(F.col("session_id") != "Unknown")
    .alias("l")
    .join(
        spark.table("lh_silver_game.sessions_clean").alias("s"),
        F.col("l.session_id") == F.col("s.session_id"),
        "left_anti"
    )
)

print(
    "Sessions tablosunda bulunmayan level event:",
    orphan_level_sessions.count()
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 19, Finished, Available, Finished, False)

Sessions tablosunda bulunmayan level event: 7813


In [18]:
unmatched_session_ids = (
    orphan_level_sessions
    .select("session_id")
    .distinct()
)

df_level_events_clean = (
    df_level_events_clean.alias("l")
    .join(
        unmatched_session_ids
        .withColumn("is_unmatched", F.lit(1))
        .alias("u"),
        on="session_id",
        how="left"
    )
    .withColumn(
        "session_id",
        F.when(
            F.col("is_unmatched") == 1,
            F.lit("Unknown")
        ).otherwise(F.col("session_id"))
    )
    .drop("is_unmatched")
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 20, Finished, Available, Finished, False)

In [19]:
orphan_level_sessions = (
    df_level_events_clean
    .filter(F.col("session_id") != "Unknown")
    .alias("l")
    .join(
        spark.table("lh_silver_game.sessions_clean").alias("s"),
        F.col("l.session_id") == F.col("s.session_id"),
        "left_anti"
    )
)

print(orphan_level_sessions.count())

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 21, Finished, Available, Finished, False)

0


In [20]:
df_level_events_clean.groupBy("booster_used").count().show()

df_level_events_clean.groupBy("unexpected_debug_flag").count().show()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 22, Finished, Available, Finished, False)

+------------+-------+
|booster_used|  count|
+------------+-------+
|        NULL|   3732|
|        true| 981293|
|       false|2752877|
+------------+-------+

+---------------------+-------+
|unexpected_debug_flag|  count|
+---------------------+-------+
|                 NULL|3734205|
|                 true|   3697|
+---------------------+-------+



In [21]:
df_level_events_clean.groupBy(
    "event_name",
    "booster_used"
).count().orderBy("event_name", "booster_used").show()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 23, Finished, Available, Finished, False)

+--------------+------------+-------+
|    event_name|booster_used|  count|
+--------------+------------+-------+
|level_complete|        NULL|   1358|
|level_complete|       false|1017168|
|level_complete|        true| 387995|
|    level_fail|        NULL|    480|
|    level_fail|       false| 359273|
|    level_fail|        true| 102677|
|   level_start|        NULL|   1894|
|   level_start|       false|1376436|
|   level_start|        true| 490621|
+--------------+------------+-------+



In [22]:
df_level_events_clean.groupBy(
    "event_name",
    "unexpected_debug_flag"
).count().orderBy("event_name", "unexpected_debug_flag").show()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 24, Finished, Available, Finished, False)

+--------------+---------------------+-------+
|    event_name|unexpected_debug_flag|  count|
+--------------+---------------------+-------+
|level_complete|                 NULL|1405173|
|level_complete|                 true|   1348|
|    level_fail|                 NULL| 461953|
|    level_fail|                 true|    477|
|   level_start|                 NULL|1867079|
|   level_start|                 true|   1872|
+--------------+---------------------+-------+



In [23]:
df_level_events_clean = (
    df_level_events_clean
    .withColumn(
        "unexpected_debug_flag",
        F.coalesce(
            F.col("unexpected_debug_flag"),
            F.lit(False)
        )
    )
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 25, Finished, Available, Finished, False)

In [24]:
df_level_events_clean.groupBy("client_build").count().show()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 26, Finished, Available, Finished, False)

+-------------+-------+
| client_build|  count|
+-------------+-------+
|0.0.0-corrupt|   3697|
|         NULL|3734205|
+-------------+-------+



In [25]:
df_level_events_clean.groupBy(
    "unexpected_debug_flag",
    "client_build"
).count().show()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 27, Finished, Available, Finished, False)

+---------------------+-------------+-------+
|unexpected_debug_flag| client_build|  count|
+---------------------+-------------+-------+
|                 true|0.0.0-corrupt|   3697|
|                false|         NULL|3734205|
+---------------------+-------------+-------+



In [26]:
df_level_events_clean = (
    df_level_events_clean
    .filter(F.col("unexpected_debug_flag") == False)
    .drop("unexpected_debug_flag", "client_build")
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 28, Finished, Available, Finished, False)

In [27]:
print("Final row count:", df_level_events_clean.count())

df_level_events_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in [
        "event_id",
        "event_name",
        "player_id",
        "session_id",
        "timestamp",
        "level_number",
        "attempt_number"
    ]
]).show()

df_level_events_clean.printSchema()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 29, Finished, Available, Finished, False)

Final row count: 3734205
+--------+----------+---------+----------+---------+------------+--------------+
|event_id|event_name|player_id|session_id|timestamp|level_number|attempt_number|
+--------+----------+---------+----------+---------+------------+--------------+
|       0|         0|        0|         0|        0|        3732|          3732|
+--------+----------+---------+----------+---------+------------+--------------+

root
 |-- session_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- attempt_number: long (nullable = true)
 |-- booster_used: boolean (nullable = true)
 |-- level_number: long (nullable = true)
 |-- moves_remaining: long (nullable = true)



In [28]:
df_level_events_clean.filter(
    F.col("level_number").isNull() |
    F.col("attempt_number").isNull()
).groupBy("event_name").count().show()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 30, Finished, Available, Finished, False)

+--------------+-----+
|    event_name|count|
+--------------+-----+
|    level_fail|  480|
|   level_start| 1894|
|level_complete| 1358|
+--------------+-----+



In [29]:
display(
    df_level_events_clean
    .filter(
        F.col("level_number").isNull() |
        F.col("attempt_number").isNull()
    )
    .limit(20)
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 61008594-afc3-4608-9894-078b770c97c0)

In [30]:
df_level_events_clean = (
    df_level_events_clean
    .filter(
        F.col("level_number").isNotNull() &
        F.col("attempt_number").isNotNull()
    )
)

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 32, Finished, Available, Finished, False)

In [31]:
print("Final row count:", df_level_events_clean.count())

df_level_events_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in [
        "event_id",
        "event_name",
        "player_id",
        "session_id",
        "timestamp",
        "level_number",
        "attempt_number"
    ]
]).show()

df_level_events_clean.printSchema()

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 33, Finished, Available, Finished, False)

Final row count: 3730473
+--------+----------+---------+----------+---------+------------+--------------+
|event_id|event_name|player_id|session_id|timestamp|level_number|attempt_number|
+--------+----------+---------+----------+---------+------------+--------------+
|       0|         0|        0|         0|        0|           0|             0|
+--------+----------+---------+----------+---------+------------+--------------+

root
 |-- session_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- attempt_number: long (nullable = true)
 |-- booster_used: boolean (nullable = true)
 |-- level_number: long (nullable = true)
 |-- moves_remaining: long (nullable = true)



In [32]:
df_level_events_clean.write.format("delta").mode("overwrite").saveAsTable("lh_silver_game.level_events_clean")

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 34, Finished, Available, Finished, False)

In [33]:
df_check = spark.table("lh_silver_game.level_events_clean")

print("Saved row count:", df_check.count())
display(df_check.limit(5))

StatementMeta(, e2fc8906-d87b-48b8-b64b-06da91280c0a, 35, Finished, Available, Finished, False)

Saved row count: 3730473


SynapseWidget(Synapse.DataFrame, ae11c316-7648-41eb-bbad-ff96cad386c7)